In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
matches = pd.read_csv("TIPE2/matches.csv", index_col=0)
matches.head()

,date,time,round,day,venue,result,gf,ga,opponent,xg,...,fk,pk,pkatt,season,team,target,venue_code,opp_code,hour,day_code
1,2021-08-15,16:30,Matchweek 1,Sun,Away,L,0.0,1.0,Tottenham,1.9,...,1.0,0.0,0.0,2022,Manchester City,0,0,18,16,6
2,2021-08-21,15:00,Matchweek 2,Sat,Home,W,5.0,0.0,Norwich City,2.7,...,1.0,0.0,0.0,2022,Manchester City,1,1,15,15,5
3,2021-08-28,12:30,Matchweek 3,Sat,Home,W,5.0,0.0,Arsenal,3.8,...,0.0,0.0,0.0,2022,Manchester City,1,1,0,12,5
4,2021-09-11,15:00,Matchweek 4,Sat,Away,W,1.0,0.0,Leicester City,2.9,...,0.0,0.0,0.0,2022,Manchester City,1,0,10,15,5
6,2021-09-18,15:00,Matchweek 5,Sat,Home,D,0.0,0.0,Southampton,1.1,...,1.0,0.0,0.0,2022,Manchester City,0,1,17,15,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38,2021-05-02,19:15,Matchweek 34,Sun,Away,L,0.0,4.0,Tottenham,0.5,...,0.0,0.0,0.0,2021,Sheffield United,0,0,18,19,6
39,2021-05-08,15:00,Matchweek 35,Sat,Home,L,0.0,2.0,Crystal Palace,0.7,...,1.0,0.0,0.0,2021,Sheffield United,0,1,6,15,5
40,2021-05-16,19:00,Matchweek 36,Sun,Away,W,1.0,0.0,Everton,1.6,...,0.0,0.0,0.0,2021,Sheffield United,1,0,7,19,6
41,2021-05-19,18:00,Matchweek 37,Wed,Away,L,0.0,1.0,Newcastle Utd,0.8,...,1.0,0.0,0.0,2021,Sheffield United,0,0,14,18,2


In [6]:
train = matches[matches["date"] < '2022-01-01']
test = matches[matches["date"] > '2022-01-01']
predictors = ["venue_code", "opp_code", "hour", "day_code"]


def sigmoid(x):
    return 1/(1+np.exp(-x))

class LogisticRegression():

    def __init__(self, lr=0.001, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iters):
            linear_pred = np.dot(X, self.weights) + self.bias
            predictions = sigmoid(linear_pred)

            dw = (1/n_samples) * np.dot(X.T, (predictions - y))
            db = (1/n_samples) * np.sum(predictions-y)

            self.weights = self.weights - self.lr*dw
            self.bias = self.bias - self.lr*db


    def predict(self, X):
        linear_pred = np.dot(X, self.weights) + self.bias
        y_pred = sigmoid(linear_pred)
        class_pred = [0 if y<=0.5 else 1 for y in y_pred]
        return class_pred
    
clf = LogisticRegression(lr=0.01)
clf.fit(train[predictors],train["target"])
y_pred = clf.predict(test[predictors])

def accuracy(y_pred, y_true):
    return np.sum(y_pred == y_true) / len(y_true)

acc = accuracy(y_pred, test["target"])
print(acc)

0.6231884057971014


In [7]:
def make_predictions(model, data, predictors):
    if not all(item in data.columns.tolist() for item in predictors):
        raise ValueError("One or more predictors are missing in the data frame.")
    preds = model.predict(data[predictors].values)
    combined = pd.DataFrame({
        "actual": data["target"],
        "predicted": preds
    }, index=data.index)
    return combined
test_set = matches[matches["date"] >= '2022-01-01']
combined_predictions = make_predictions(clf, test_set, predictors)
combined_predictions = combined_predictions.merge(test_set[["date", "team", "opponent", "result"]], left_index=True, right_index=True)
print(combined_predictions.head())


    actual  predicted       date                     team   opponent result
29       1          0 2022-01-01          Manchester City    Arsenal      W
29       1          0 2022-02-19                  Arsenal  Brentford      W
29       1          0 2022-01-22        Manchester United   West Ham      W
29       1          0 2022-02-27  Wolverhampton Wanderers   West Ham      L
29       1          0 2022-03-13         Newcastle United    Chelsea      L
